# 05 · Coverage simulation: Fisher vs cluster bootstrap vs mixed-effects intervals

C2 simulation on the Stage-1 FF calendar design (33 cells, 11 conditions, same check-up times). Each replicate redraws cell effects $b_c \sim N(0,\tau^2)$ and noise, refits the per-SoC-level model of `04_calendar_fit.ipynb`, and builds 95% intervals from:

- **Fisher**: Gauss–Newton covariance from least squares with a pooled residual variance;
- **cluster bootstrap**: resampling cells within condition ($B = 500$), Wald and percentile;
- **mixed Wald**: inverse numerical Hessian of the marginal likelihood (cell random effect on $\log k$).

Two scenarios, differing in what generates the data:

- **well-specified**: truth is the per-SoC mixed fit ($\tau \approx 0.21$, $\sigma \approx 0.0014$); the analysis model is correct;
- **misspecified**: truth is the *saturated* mixed fit (free rate per condition, $\tau \approx 0.036$), analyzed with the per-SoC model. The target is the pseudo-true parameter, the least-squares projection of the noise-free saturated curves onto the per-SoC model.

The simulation is `julia/simulate_coverage.jl` (threaded, about 15 s for 500 replicates). The Python version `src/simulate_coverage.py` implements the well-specified scenario only and is about 20× slower.

In [1]:
# Run both scenarios with the Julia script (overwrites ../data/processed/sim_coverage_soc*.parquet)
# ~15 s per scenario for 500 replicates on 14 cores (plus Julia startup/compilation); rerun the cells below afterwards.
import shutil, subprocess
from pathlib import Path

REPS = 500   # simulation replicates
B = 500      # bootstrap draws per replicate

julia = shutil.which("julia") or str(Path.home() / ".juliaup" / "bin" / "julia")
for scenario in ["wellspec", "misspec"]:
    # Run from the repo root (the project path is relative to it); stream output as it runs.
    # The parquet is written only when all replicates finish.
    proc = subprocess.Popen([julia, "-t", "auto", "--project=julia", "julia/simulate_coverage.jl",
                             str(REPS), str(B), scenario],
                            cwd="..", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"{scenario} simulation exited with code {proc.returncode}")

design: 165 rows, 33 cells, 11 conditions | scenario: wellspec | threads: 14


generating model: sigma=0.00138 tau=0.2056


  100 / 500 replicates (3 s)


  200 / 500 replicates (6 s)


  300 / 500 replicates (8 s)


  400 / 500 replicates (10 s)


... (6 outputs left out)

  300 / 500 replicates (8 s)


  400 / 500 replicates (10 s)


  500 / 500 replicates (13 s)


saved 500 / 500 replicates to /Users/chenweihu/code/thesis-oed/data/processed/sim_coverage_soc_misspec.parquet (15 s)


## Scenario 1: well-specified

In [2]:
# Empirical SE vs mean estimated SE, and 95% coverage, per parameter (excluding diverged replicates)
import sys
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
from scipy import stats

s = pd.read_parquet("../data/processed/sim_coverage_soc.parquet")
r = s.groupby("rep")[["tau_hat", "sigma_hat"]].first()
bad = r[~np.isfinite(r["tau_hat"]) | (r["tau_hat"] > 1) | (r["sigma_hat"] < 5e-4) | (r["sigma_hat"] > 5e-3)].index
g0 = s[~s["rep"].isin(bad)]

def cover(lo, hi, t):
    return ((lo <= t) & (t <= hi)).mean()

rows = []
for p, g in g0.groupby("param", sort=False):
    t = g["truth"].iloc[0]
    rows.append({"param": p, "n": len(g),
                 "SD(est) NLS": g["est_nls"].std(), "SD(est) mixed": g["est_mixed"].std(),
                 "Fisher SE": g["se_fisher"].mean(), "boot SE": g["se_boot"].mean(), "mixed SE": g["se_mixed"].mean(),
                 "cov Fisher": cover(g["est_nls"] - 1.96 * g["se_fisher"], g["est_nls"] + 1.96 * g["se_fisher"], t),
                 "cov boot Wald": cover(g["est_nls"] - 1.96 * g["se_boot"], g["est_nls"] + 1.96 * g["se_boot"], t),
                 "cov boot pct": cover(g["pct_lo"], g["pct_hi"], t),
                 "cov mixed": cover(g["est_mixed"] - 1.96 * g["se_mixed"], g["est_mixed"] + 1.96 * g["se_mixed"], t)})
tab = pd.DataFrame(rows).set_index("param")
tab.round(3)

,n,SD(est) NLS,SD(est) mixed,Fisher SE,boot SE,mixed SE,cov Fisher,cov boot Wald,cov boot pct,cov mixed
param,,,,,,,,,,
log_k_ref,500,0.087,0.074,0.052,0.071,0.066,0.740,0.838,0.834,0.916
Ea_kJ,500,2.980,2.450,1.094,2.170,2.017,0.516,0.802,0.800,0.886
delta_soc0.05,500,0.129,0.115,0.118,0.104,0.105,0.926,0.846,0.842,0.934
delta_soc0.2,500,0.153,0.147,0.089,0.122,0.137,0.750,0.844,0.836,0.926
delta_soc0.8,500,0.155,0.149,0.048,0.120,0.133,0.466,0.814,0.806,0.912
delta_soc1.0,500,0.119,0.098,0.045,0.093,0.089,0.544,0.846,0.832,0.916
z,500,0.047,0.035,0.079,0.037,0.034,1.000,0.876,0.868,0.944


**Findings** (from the 100-replicate run: 95 usable replicates; coverage Monte Carlo SE is about 0.02–0.03).
- Fisher intervals badly undercover the rate parameters (about 0.45–0.55 for $E_a$ and the SoC 0.8 and 1.0 effects) and overcover $z$ (0.99): the pooled residual variance ignores cell clustering.
- The cluster bootstrap undercovers uniformly at about 0.80–0.88, consistent with only 3 cells per condition.
- Mixed Wald intervals are closest to nominal (0.85–0.95) when the model is correct.
- Estimated τ averages about 0.18 against a truth of 0.206 (ML is biased low with 33 cells).

**Limits.** The mixed-model optimizer diverged in 5 of 100 replicates (no bounds on log σ, log τ); these are excluded, not fixed. Only the well-specified scenario has been run. Next: bounds and restarts, 500 replicates, a misspecified scenario (simulate from the saturated mixed model, fit the per-SoC model), and profile-likelihood intervals.

## Scenario 2: misspecified (saturated truth, per-SoC analysis model)

In [3]:
# Misspecified scenario: bias, SEs, and coverage of the pseudo-true parameter
import numpy as np
import pandas as pd

s = pd.read_parquet("../data/processed/sim_coverage_soc_misspec.parquet")
r = s.groupby("rep")[["tau_hat", "sigma_hat"]].first()
bad = r[~np.isfinite(r["tau_hat"]) | (r["tau_hat"] > 1) | (r["sigma_hat"] < 5e-4) | (r["sigma_hat"] > 5e-3)].index
print(f"replicates: {r.shape[0]} | diverged mixed fits excluded: {len(bad)}")
print(f"tau_hat mean {r['tau_hat'].mean():.3f} (generating tau 0.036) | sigma_hat mean {r['sigma_hat'].mean():.5f}")
g0 = s[~s["rep"].isin(bad)]

def cover(lo, hi, t):
    return ((lo <= t) & (t <= hi)).mean()

rows = []
for p, g in g0.groupby("param", sort=False):
    t = g["truth"].iloc[0]
    rows.append({"param": p, "target": t,
                 "bias NLS": g["est_nls"].mean() - t, "bias mixed": g["est_mixed"].mean() - t,
                 "SD(est) NLS": g["est_nls"].std(), "SD(est) mixed": g["est_mixed"].std(),
                 "Fisher SE": g["se_fisher"].mean(), "boot SE": g["se_boot"].mean(), "mixed SE": g["se_mixed"].mean(),
                 "cov Fisher": cover(g["est_nls"] - 1.96 * g["se_fisher"], g["est_nls"] + 1.96 * g["se_fisher"], t),
                 "cov boot Wald": cover(g["est_nls"] - 1.96 * g["se_boot"], g["est_nls"] + 1.96 * g["se_boot"], t),
                 "cov boot pct": cover(g["pct_lo"], g["pct_hi"], t),
                 "cov mixed": cover(g["est_mixed"] - 1.96 * g["se_mixed"], g["est_mixed"] + 1.96 * g["se_mixed"], t)})
pd.DataFrame(rows).set_index("param").round(3)

replicates: 500 | diverged mixed fits excluded: 0
tau_hat mean 0.186 (generating tau 0.036) | sigma_hat mean 0.00141


,target,bias NLS,bias mixed,SD(est) NLS,SD(est) mixed,Fisher SE,boot SE,mixed SE,cov Fisher,cov boot Wald,cov boot pct,cov mixed
param,,,,,,,,,,,,
log_k_ref,-3.671,0.001,-0.070,0.023,0.028,0.030,0.018,0.068,0.982,0.860,0.870,0.978
Ea_kJ,23.115,-0.068,-2.984,0.758,1.076,0.751,0.566,2.093,0.940,0.840,0.828,0.906
delta_soc0.05,-1.266,-0.002,0.174,0.055,0.058,0.083,0.045,0.107,0.996,0.854,0.846,0.722
delta_soc0.2,-0.700,0.000,0.166,0.046,0.053,0.059,0.036,0.141,0.986,0.812,0.776,0.984
delta_soc0.8,0.439,0.001,0.172,0.035,0.044,0.029,0.027,0.137,0.892,0.820,0.806,1.000
delta_soc1.0,0.340,0.001,0.115,0.029,0.031,0.028,0.022,0.092,0.934,0.822,0.812,0.974
z,0.728,0.001,-0.052,0.033,0.034,0.053,0.026,0.035,0.996,0.854,0.840,0.698


## Findings (500 replicates per scenario; coverage Monte Carlo SE about 0.01–0.02)

**Well-specified.**
- Fisher intervals badly undercover the rate parameters (0.47–0.54 for the SoC 0.8 and 1.0 effects, 0.52 for $E_a$) and overcover $z$ (1.00): the pooled residual variance ignores cell clustering.
- The cluster bootstrap undercovers uniformly at about 0.80–0.88, consistent with only 3 cells per condition.
- Mixed Wald intervals are closest to nominal (0.89–0.94), still slightly low. $\hat\tau$ averages 0.18 against a truth of 0.206 (ML is biased low with 33 cells).

**Misspecified.**
- $\hat\tau$ averages 0.19 although the generating $\tau$ is 0.036: the per-SoC model's random effect absorbs condition-level misfit. This reproduces the real-data signature ($\hat\tau = 0.206$ per-SoC vs 0.036 saturated, notebook 04).
- NLS is unbiased for the least-squares pseudo-truth by construction. Fisher intervals overcover (0.89–1.00) because the pooled residual variance now includes the misfit; the bootstrap undercovers (0.81–0.87).
- The mixed estimator is shifted relative to the least-squares target (about −3.0 kJ for $E_a$, +0.12 to +0.17 for the SoC effects, −0.05 for $z$). These are the same shifts seen between the NLS and mixed fits on the real data (−3.2 kJ, +0.12 to +0.19, −0.07), so that real-data discrepancy is consistent with misspecification and not with a better fit.
- "Coverage" of the mixed interval against the least-squares target (0.70–1.00, erratic) is not interpretable: with a wrong mean structure it estimates a different pseudo-true parameter. Judging it needs a common target or a prediction-based criterion.

**Limits.** One misspecification (saturated truth), one design, no profile-likelihood intervals. The Julia mixed fit bounds $\log\sigma$ and $\log\tau$; no replicate diverged or hit a bound. The pseudo-truth for the mixed estimator is not computed separately.